# **Sesión 3:** Procesamiento de imagenes y extracción de características

## **Librerías**

In [ ]:
import cv2
print("OpenCV should be 4.8.0.76 Current version:", cv2.__version__)
from typing import List
import numpy as np
import imageio
import copy
import glob
import matplotlib.pyplot as plt
from typing import Optional
from utils import *

## **Apartado A:** Filtro Gaussiano y Detección de bordes: Sobel y Canny

El objetivo de este apartado es detectar los bordes de las imágenes de la carpeta ``data/partA-B``. Para ello, deberá seguir los siguientes pasos:

1. **Tarea A.1.** Defina el método ``gaussian_blur()`` que aplique un filtro gausiano para obtener imágenes borrosas. Siga todas las indicaciones del enunciado.
2. **Tarea A.2.** Aplique el método ``gaussian_blur()`` a todas las imágenes en ``data/partA-B``.


3. **Tarea A.3.** Defina la función ``sobel_edge_detector()`` que detecte bordes con el método Sobel. Siga todas las indicaciones del enunciado.
4. **Tarea A.4.** Aplique el método ``sobel_edge_detector()`` a todas las imágenes en ``data/partA-B``.


5. **Tarea A.5.** Defina la función ``canny_edge_detector()`` que detecte bordes con el método Canny. Siga todas las indicaciones del enunciado.
6. **Tarea A.6.** Aplique el método ``canny_edge_detector()`` a todas las imágenes en ``data/partA-B``.

### **Tarea PREVIA:** Defina y ejecute el método para cargar imágenes ``load_images()``

In [ ]:
def load_images(filenames: List) -> List:
    return [cv2.imread(filename) for filename in filenames]

# Build a list containing the paths of all images in the data folder
imgs_path = sorted(glob.glob("../data/partA-B/*.jpg"))
imgs = load_images(imgs_path)

### **Tarea PREVIA:** Defina la función ``show_image()``

In [ ]:
def show_image(img: np.array, img_name: str = "Image"):
    # Show the image
    cv2.imshow(img_name, img)
    # Wait until any key is pressed
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [ ]:
show_image(imgs[0])

### **Tarea A.1:** Defina el método ``gaussian_blur()`` que aplique un filtro gausiano para obtener imágenes borrosas.

In [ ]:
# TODO Define the method
def gaussian_blur(img: np.array, sigma: float, filter_shape: Optional[List] = None, verbose: bool = False) -> np.array:
    # TODO If not given, compute the filter shape 
    if filter_shape is None:
        radius = int(np.ceil(4*sigma)) # asegurar que es impar 
        filter_l = 2*radius + 1 #f(sigma)
    else:
        # me dan el filtro
        filter_l = filter_shape[0]
        radius = filter_l // 2
    
    # TODO Create the filter coordinates matrices
    y, x = np.mgrid[-radius:radius+1, -radius:radius+1]
    
    # TODO Define the formula that goberns the filter
    formula = 1/(2*np.pi*sigma**2) * np.exp(- (x**2 + y**2)/(2*sigma**2))
    
    gaussian_filter = formula / np.sum(formula) # Normalize filter: preserve shine
    
    # TODO Process the image
    gb_img = cv2.filter2D(img, -1,  gaussian_filter)
    
    if verbose:
        show_image(img=gb_img, img_name=f"Gaussian Blur: Sigma = {sigma}")
    
    return gaussian_filter, gb_img.astype(np.uint8)

In [ ]:
# TEST
radius = 3 # asegurar que es impar 
filter_l = 2*radius + 1 #f(sigma)
    
    
# TODO Create the filter coordinates matrices
y_test, x_test = np.mgrid[-radius:radius+1, -radius:radius+1] #aisla dos vectores
x_test
    

### **Tarea A.2.** Aplique el método ``gaussian_blur()`` a todas las imágenes en ``data``.

In [ ]:
# TODO Get the gaussian blurred images using a list comprehension
gauss_sigma = 2
gb_imgs = [gaussian_blur(img, gauss_sigma)[1] for img in imgs]

show_image(gb_imgs[4])

### **Tarea A.3:** Defina la función ``sobel_edge_detector()`` que detecte bordes con el método Sobel.

In [ ]:
# TODO Define the method


def sobel_edge_detector(img: np.array, filter: np.array, gauss_sigma: float, gauss_filter_shape: Optional[List] = None, verbose: bool = False) -> np.array:
    # TODO Transform the img to grayscale
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # TODO Get a blurry img to improve edge detections
    _, blurred = gaussian_blur(img=gray_img, sigma=gauss_sigma, filter_shape=gauss_filter_shape, verbose=verbose)
    
    # Re-scale
    blurred = blurred/255
    
    # TODO Get vertical edges
    v_edges = cv2.filter2D(blurred, -1,  filter)
    
    # TODO Transform the filter to get the orthogonal edges
    filter = filter.T 
    
    # TODO Get horizontal edges
    h_edges = cv2.filter2D(blurred, -1,  filter)
    
    # TODO Get edges
    sobel_edges_img = np.hypot(h_edges, v_edges)
    
    # Get edges angle
    theta = np.arctan2(v_edges, h_edges)
    
    # Visualize if needed
    if verbose:
        show_image(img=sobel_edges_img, img_name="Sobel Edges")
    
    return np.squeeze(sobel_edges_img), np.squeeze(theta)

### **Tarea A.4.** Aplique el método ``sobel_edge_detector()`` a todas las imágenes en ``data``.

In [ ]:
# TODO Define a sigma value
gauss_sigma = 1

# TODO Define the Sobel filter
sobel_filter = np.array([
    [-1,  0,  1],
    [-2,  0,  2],
    [-1,  0,  1]
])
# TODO Get the edges detected by Sobel using a list comprehension
sobel_edges_imgs = [sobel_edge_detector(img, sobel_filter, gauss_sigma, None, True)[0] for img in imgs]

### **Tarea A.5:** Defina la función ``canny_edge_detector()`` que detecte bordes con el método Canny.

In [ ]:
# TODO Define the method
def canny_edge_detector(img: np.array, sobel_filter: np.array, gauss_sigma: float, gauss_filter_shape: Optional[List] = None, verbose: bool = False):
    # TODO Call the method sobel_edge_detector()
    sobel_edges_img, theta = sobel_edge_detector(img, sobel_filter, gauss_sigma, gauss_filter_shape, verbose)
    
    # TODO Use NMS to refine edges
    canny_edges_img = non_max_suppression(sobel_edges_img, theta)
    
    # Calculamos un umbral relativo (por ejemplo, el 15% del valor más alto)
    umbral = 0.25 * np.max(canny_edges_img)
    
    # Todo lo que supere el umbral se vuelve 255 (blanco), el resto 0 (negro)
    canny_edges_img = np.where(canny_edges_img > umbral, 255, 0).astype(np.uint8)
    
    if verbose:
        show_image(canny_edges_img, img_name="Canny Edges")
        
    return canny_edges_img

### **Tarea A.6.** Aplique el método ``canny_edge_detector()`` a todas las imágenes en ``data``.

In [ ]:
# TODO Define Sobel filter
sobel_filter = np.array([
    [-1,  0,  1],
    [-2,  0,  2],
    [-1,  0,  1]
])

# TODO Define a sigma value for Gauss
gauss_sigma = 2

# TODO Define a Gauss filter shape
gauss_filter_shape = None

# TODO Get the edges detected by Canny using a list comprehension
canny_edges = [canny_edge_detector(img, sobel_filter, gauss_sigma, gauss_filter_shape, True) for img in imgs]

### **Pregunta A.1:** Añada ruido a las imágenes de la carpeta ``data``. Compare los resultados que obtiene al aplicar su filtro Sobel con y sin filtro Gausiano.

In [ ]:
# TODO Homework

def sobel_edge_without_gaus(img, filter):
    gray_ruidosa = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) / 255.0 #normalizing it
    v_edges_raw = cv2.filter2D(gray_ruidosa, -1, filter)
    trans_filter = filter.T
    h_edges_raw = cv2.filter2D(gray_ruidosa, -1, trans_filter)
    
    sobel_edges_img = np.hypot(h_edges_raw, v_edges_raw)
    
    return sobel_edges_img
    
for img in imgs:
    img_x = img.copy()

    # 2. Generamos la matriz de ruido gaussiano
    # np.random.normal(media, desviacion_estandar, tamaño_matriz)
    noise_matrix = np.random.normal(0, 30, img_x.shape)

    noisy_img = img_x + noise_matrix

    # 4. CRÍTICO: Recortamos los valores que se salgan de [0, 255] y convertimos a uint8
    noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)


    show_image(noisy_img, "Imagen con Ruido")


    sobel_filter = np.array([
        [-1,  0,  1],
        [-2,  0,  2],
        [-1,  0,  1]
    ])

    sobel_without_gauss = sobel_edge_without_gaus(noisy_img, sobel_filter)

    show_image(sobel_without_gauss, "Sobel sin filtro Gaussiano ")


    gauss_sigma = 2
    sobel_with_gauss, _ = sobel_edge_detector(noisy_img, sobel_filter, gauss_sigma)

    show_image(sobel_with_gauss, "Sobel con Gaussiano")

### **Pregunta A.2:** Utilice la librería ``scikit-image`` y compare el efecto de los filtros Sobel, Canny y Prewitt sobre las imágenes de la carpeta ``data``. ¿Qué diferencias observa entre los filtros? ¿Puede obtener alguna conclusión y/o patrón?

In [ ]:
# TODO Homework
from skimage import filters, feature

i = 0
# 1. Seleccionamos la primera imagen y la pasamos a grises (skimage lo requiere)
for img in imgs:
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # applying each filter:
    edges_prewitt = filters.prewitt(img_gray)
    edges_sobel = filters.sobel(img_gray)
    edges_canny = feature.canny(img_gray, sigma=2.0) # Permite ajustar el sigma integrado
    # edges_canny = feature.canny(img_gray, sigma=2.0, low_threshold=0.1, high_threshold=0.3)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    fig.suptitle(f"Imagen 0{i}", fontsize=18, fontweight='bold')

    axes[0].imshow(edges_prewitt, cmap='gray')
    axes[0].set_title("Filtro Prewitt")
    axes[0].axis('off')

    axes[1].imshow(edges_sobel, cmap='gray')
    axes[1].set_title("Filtro Sobel")
    axes[1].axis('off')

    axes[2].imshow(edges_canny, cmap='gray')
    axes[2].set_title("Filtro Canny")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()
    
    i += 1

### **Pregunta A.3:** Añada ruido impulsivo ("sal y pimienta") a una imagen de la carpeta ``data`` corrompiendo un 5 % de los píxeles. Aplique sobre la imagen ruidosa un filtro gaussiano (``cv2.GaussianBlur()``) y uno de mediana (``cv2.medianBlur()``) y explique por qué se obtienen resultados distintos.

In [ ]:
# TODO Homework

def add_salt_and_pepper(img: np.array, amount: float = 0.05, seed: Optional[int] = 0) -> np.array:
    rng = np.random.default_rng(seed) # generador de números aleatorios con semilla
    noisy = img.copy()
    h, w = img.shape[:2]

    n_corrupt = int(round(amount * h * w)) # cuantos píxeles sacar en total : amount%
    idx = rng.choice(h * w, size=n_corrupt, replace=False) # pixeles random
    rows, cols = np.unravel_index(idx, (h, w))

    # Para cada píxel elegido: 0 o 255 con probabilidad 0.5
    values = rng.choice([0, 255], size=n_corrupt).astype(np.uint8)
    # nueva imagen con ruido
    noisy[rows, cols] = values[:, None] if img.ndim == 3 else values  # mismo valor en B, G y R

    return noisy

In [ ]:
original = imgs[0]
noisy = add_salt_and_pepper(original, amount=0.05)

# Aplicar filtro sobre la imagen ruidosa
ksize = 5  # debe ser impar 
gauss_res = cv2.GaussianBlur(noisy, (ksize, ksize), 0)
median_res = cv2.medianBlur(noisy, ksize)

results = [
    ("Original", original),
    ("Ruido sal y pimienta (5 %, kernel=5)", noisy),
    (f"Gaussiano {ksize}x{ksize}", gauss_res),
    (f"Mediana {ksize}x{ksize}", median_res),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
for ax, (title, im) in zip(axes, results):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title}")
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ESTUDIO: tamaño del kernel (ruido fijo 5%)
kernel_sizes = [3, 5, 9, 15]

original = imgs[0]
noisy = add_salt_and_pepper(original, amount=0.05)

fig, axes = plt.subplots(2, len(kernel_sizes), figsize=(5 * len(kernel_sizes), 9))
for col, k in enumerate(kernel_sizes):
    g = cv2.GaussianBlur(noisy, (k, k), 0)
    m = cv2.medianBlur(noisy, k)
    axes[0, col].imshow(cv2.cvtColor(g, cv2.COLOR_BGR2RGB))
    axes[0, col].set_title(f"Gaussiano k={k} ")
    axes[1, col].imshow(cv2.cvtColor(m, cv2.COLOR_BGR2RGB))
    axes[1, col].set_title(f"Mediana k={k}")
    axes[0, col].axis('off'); axes[1, col].axis('off')
plt.tight_layout()
plt.show()

## **Apartado B:** Operadores Morfológicos

Para resolver este partado, deberá seguir los siguientes pasos:

1. **Tarea B.1.** Defina el método ``binarize()`` para binarizar imágenes.
2. **Tarea B.2.** Defina el método ``custom_dilate()``.
3. **Tarea B.3.** Defina el método ``custom_erode()``.
4. **Pregunta B.1** Aplique los métodos ``custom_dilate()`` y ``custom_erode()`` a todas las imágenes de la carpeta ``data``.


### **Tarea B.1.** Defina el método ``binarize()`` para binarizar imágenes.

In [ ]:
# TODO Homework: define the binarization method
def binarize(img: np.array, threshold: int = 127):
    # filtro gris
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # comparar pixel con umbral
    binary_img = np.where(gray > threshold, 255, 0).astype(np.uint8)
    return binary_img

### **Tarea B.2.** Defina el método ``custom_dilate()``

In [ ]:
# TODO Homework: define the dilation method
def custom_dilate(img):
    # Structuring element 3x3 con valores 255 (blanco)
    kernel = np.full((3, 3), 255, dtype=np.uint8)

    # TODO pad the original image so it can keep dimensions after processing
    # añadir un borde alrededor del arrary (img) = 0 (negro) = no dilata nada
    padded = np.pad(img, pad_width=1, mode='constant', constant_values=0)

    # TODO get img shape
    height, width = img.shape[:2]
    
    # TODO Create an element with the same dimensions as the padded img
    dilated = np.zeros_like(padded)
    
    for j in range(height):
        for i in range(width):
            # TODO Add logic to the operation
            window = padded[j:j+3, i:i+3] # Ventana 3x3 centrada en el píxel (j, i) de la imagen

            # DILATACION
            # el píxel pasa a blanco si window toca algún píxel blanco
            if np.any((window == 255) & (kernel == 255)):
                dilated[j+1, i+1] = 255
            
    # TODO Select the region of interest (ROI). Modify if needed
    dilated = dilated[1:height+1, 1:width+1]
    
    return dilated

### **Tarea B.3.** Defina el método ``custom_erode()``

In [ ]:
# TODO Homework: define the erotion method
def custom_erode(img):
    # Structuring element 3x3 con valores 255 (blanco)
    kernel = np.full((3, 3), 255, dtype=np.uint8)

    # TODO pad the original image so it can keep dimensions after processing
    padded = np.pad(img, pad_width=1, mode='constant', constant_values=255)

    # TODO get img shape
    height, width = img.shape[:2]
    
    # TODO Create an element with the same dimensions as the padded img
    eroded = np.zeros_like(padded)
    
    for j in range(height):
        for i in range(width):
            # TODO Add logic to the operation
            window = padded[j:j+3, i:i+3] # Ventana 3x3 centrada en el píxel (j, i) de la imagen

            # ELOCIÓN
            if np.all(window[kernel == 255] == 255):
                eroded[j+1, i+1] = 255
            
    # TODO Select the region of interest (ROI). Modify if needed
    eroded = eroded[1:height+1, 1:width+1]
    
    return eroded

### **Pregunta B.1** Aplique los métodos ``custom_dilate()`` y ``custom_erode()`` a todas las imágenes de la carpeta ``data``.

In [ ]:
# TODO Homework

# para las funciones de cv2
kernel_cv = np.full((3, 3), 255, dtype=np.uint8)


for n, img in enumerate(imgs):
    # binarizar
    binary = binarize(img)

    # implementaciones
    dilated = custom_dilate(binary)
    eroded = custom_erode(binary)

    # COMPARACIÓN
    ok_dil = np.array_equal(dilated, cv2.dilate(binary, kernel_cv))
    ok_ero = np.array_equal(eroded, cv2.erode(binary, kernel_cv))
    print(f"Imagen {n:02d} -> dilatación == cv2.dilate: {ok_dil} | erosión == cv2.erode: {ok_ero}")

    fig, axes = plt.subplots(1, 4, figsize=(22, 6))
    fig.suptitle(f"Imagen 0{n}", fontsize=16, fontweight='bold')
    for ax, (title, im, cmap) in zip(axes, [
        ("Original", cv2.cvtColor(img, cv2.COLOR_BGR2RGB), None),
        ("Binarizada (umbral 127)", binary, 'gray'),
        ("Dilatación", dilated, 'gray'),
        ("Erosión", eroded, 'gray'),
    ]):
        ax.imshow(im, cmap=cmap, vmin=0, vmax=255) if cmap else ax.imshow(im)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
